In [1]:
import cv2
import torch
from moge.model.v3 import MoGeModel

torch.cuda.is_available()

True

In [7]:
ckpt = torch.load("model.pt", map_location="cpu", weights_only=True)
print(list(ckpt.keys()))                    # should include 'model' and 'model_config'
print(list(ckpt["model_config"].keys()))     # <-- check if 'neck' is missing here

['model_config', 'model']
['encoder', 'remap_output', 'output_mask', 'split_head', 'intermediate_layers', 'dim_upsample', 'dim_times_res_block_hidden', 'num_res_blocks', 'trained_area_range', 'last_conv_channels', 'last_conv_size']


In [8]:
from huggingface_hub import hf_hub_download

In [9]:
path = hf_hub_download(repo_id="Ruicheng/moge-3-vitl", filename="model.pt", local_dir=".")
print(path)

model.pt: reconstructing file:   0%|          |  0.00B / 1.48GB            

model.pt: downloading bytes:           |  0.00B            

F:\Documents\LiDAR_kappazunder_stadpark_reduced\MoGe3_pipeline\model.pt


In [10]:
ckpt = torch.load("model.pt", map_location="cpu", weights_only=True)
print(list(ckpt.keys()))                    # should include 'model' and 'model_config'
print(list(ckpt["model_config"].keys()))     # <-- check if 'neck' is missing here

['model_config', 'model']
['encoder', 'neck', 'points_head', 'normal_head', 'mask_head', 'scale_head', 'num_tokens_range', 'refiner', 'refiner_depth_resolution']


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
	print(f"Using device: {torch.cuda.get_device_name(device)}")
else:
	print("Using device: CPU")

model = MoGeModel.from_pretrained("model.pt").to(device)

Using device: NVIDIA GeForce RTX 4070
Using device: NVIDIA GeForce RTX 4070


In [ ]:
input_image = cv2.cvtColor(cv2.imread("../)

`moge infer -i ..\colmap_pipeline\RS_export\images --version v3 --pretrained model.pt --refine_steps 3 -o .\output --maps --resize 2 --fov_x 90`

In [6]:
# The above CLI command created normal and depth maps in ./output/frame_*/depth_vis.png and ./output/frame_*/normal.png. Copy them into ../colmap_pipeline/RS_export/normal/frame_*.png and ../colmap_pipeline/RS_export/depth/frame_*.png, respectively. Make sure to copy only the frames that are present in ../colmap_pipeline/RS_export/images/frame_*.jpg.
import os
import shutil
import cv2

export_path = "../colmap_pipeline/RS_export"
images_path = os.path.join(export_path, "images")
depth_output_path = os.path.join(export_path, "depth")
normal_output_path = os.path.join(export_path, "normal")

In [7]:

# Create the output directories if they don't exist
os.makedirs(depth_output_path, exist_ok=True)
os.makedirs(normal_output_path, exist_ok=True)
# Clear the output directories before copying new files
for output_path in [depth_output_path, normal_output_path]:
	for file in os.listdir(output_path):
		file_path = os.path.join(output_path, file)
		if os.path.isfile(file_path):
			os.remove(file_path)

# Copy the depth and normal maps to the respective directories if they exist in images_path
for frame_file in os.listdir(images_path):
	frame_name, ext = os.path.splitext(frame_file)
	if ext.lower() == ".jpg":
		depth_map_path = os.path.join("./output", frame_name, "depth_vis.png")
		normal_map_path = os.path.join("./output", frame_name, "normal.png")

		if os.path.exists(depth_map_path):
			shutil.copy(depth_map_path, os.path.join(depth_output_path, f"{frame_name}.png"))
			print(f"Copied depth map for {frame_name}")
		else:
			print(f"Depth map not found for {frame_name}")

		if os.path.exists(normal_map_path):
			# Instead of copying, read the normal map, invert Y and Z channels, and save it to the normal output path
			normal_map = cv2.imread(normal_map_path, cv2.IMREAD_UNCHANGED)
			if normal_map is not None:
				# Invert the Y and Z channels (assuming normal map is in RGB format)
				inverted_normal_map = normal_map.copy()
				# inverted_normal_map[:, :, 0] = 255 - inverted_normal_map[:, :, 0]  # Invert X channel
				# inverted_normal_map[:, :, 1] = 255 - inverted_normal_map[:, :, 1]  # Invert Y channel
				cv2.imwrite(os.path.join(normal_output_path, f"{frame_name}.png"), inverted_normal_map)
				print(f"Inverted and saved normal map for {frame_name}")
			# shutil.copy(normal_map_path, os.path.join(normal_output_path, f"{frame_name}.png"))
			# print(f"Copied normal map for {frame_name}")
		else:
			print(f"Normal map not found for {frame_name}")

Copied depth map for frame_000002
Inverted and saved normal map for frame_000002
Copied depth map for frame_000003
Inverted and saved normal map for frame_000003
Copied depth map for frame_000007
Inverted and saved normal map for frame_000007
Copied depth map for frame_000008
Inverted and saved normal map for frame_000008
Copied depth map for frame_000011
Inverted and saved normal map for frame_000011
Copied depth map for frame_000013
Inverted and saved normal map for frame_000013
Copied depth map for frame_000017
Inverted and saved normal map for frame_000017
Copied depth map for frame_000018
Inverted and saved normal map for frame_000018
Copied depth map for frame_000021
Inverted and saved normal map for frame_000021
Copied depth map for frame_000023
Inverted and saved normal map for frame_000023
Copied depth map for frame_000024
Inverted and saved normal map for frame_000024
Copied depth map for frame_000026
Inverted and saved normal map for frame_000026
Copied depth map for frame_0

In [5]:
# MoGe additionally created for each frame an automatic mask which includes the sky and (sometimes) the car hood in black and the foreground in white. We already have reliable car hood masks in ../colmap_pipeline/RS_export/mask_car/frame_*.jpg. We thus want to combine the upper half of the automatic sky masks (./output/frame_*/mask.png) with the lower half of the car hood masks (../colmap_pipeline/RS_export/mask_car/frame_*.jpg) to create a new mask for each frame. The new masks should be saved in ../colmap_pipeline/RS_export/masks/frame_*.png.
export_mask_path = os.path.join(export_path, "masks")
car_mask_path = os.path.join(export_path, "masks_car")
os.makedirs(export_mask_path, exist_ok=True)
# Clear the output mask directory before copying new files
for file in os.listdir(export_mask_path):
	file_path = os.path.join(export_mask_path, file)
	if os.path.isfile(file_path):
		os.remove(file_path)

for frame_file in os.listdir(images_path):
	frame_name, ext = os.path.splitext(frame_file)

	if ext.lower() != ".jpg":
		continue

	automatic_mask_path = os.path.join("./output", frame_name, "mask.png")
	car_mask_path_frame = os.path.join(car_mask_path, frame_file)

	automatic_mask = cv2.imread(automatic_mask_path, cv2.IMREAD_GRAYSCALE)

	if automatic_mask is None:
		print(f"Automatic mask not found for {frame_name}")
		continue

	# Invert the automatic sky mask
	new_mask = 255 - automatic_mask
	height, width = new_mask.shape

	if os.path.exists(car_mask_path_frame):
		car_mask = cv2.imread(car_mask_path_frame, cv2.IMREAD_GRAYSCALE)

		if car_mask is not None:
			car_mask = cv2.resize(car_mask, (width, height))
			new_mask[height // 2:] = car_mask[height // 2:]
		else:
			new_mask[height // 2:] = 0
			print(f"Could not read car mask for {frame_name}; bottom set to black")
	else:
		new_mask[height // 2:] = 0
		print(f"Car mask not found for {frame_name}; bottom set to black")

	cv2.imwrite(os.path.join(export_mask_path, f"{frame_name}.png"), new_mask)
	print(f"Created combined mask for {frame_name}")

Car mask not found for frame_000002; bottom set to black
Created combined mask for frame_000002
Created combined mask for frame_000003
Car mask not found for frame_000007; bottom set to black
Created combined mask for frame_000007
Created combined mask for frame_000008
Car mask not found for frame_000011; bottom set to black
Created combined mask for frame_000011
Created combined mask for frame_000013
Car mask not found for frame_000017; bottom set to black
Created combined mask for frame_000017
Created combined mask for frame_000018
Created combined mask for frame_000021
Car mask not found for frame_000023; bottom set to black
Created combined mask for frame_000023
Car mask not found for frame_000024; bottom set to black
Created combined mask for frame_000024
Created combined mask for frame_000026
Car mask not found for frame_000027; bottom set to black
Created combined mask for frame_000027
Car mask not found for frame_000028; bottom set to black
Created combined mask for frame_00002